<h1><b><i>SCRIPT PRINCIPAL</b></i></h1>
<h2><b><i>Importação de Bibliotécas</b></i></h2>

In [55]:
import sys
sys.path.append(r'C:\pylibs')

import pandas as pd
import numpy as np
import openpyxl

In [56]:
import pandas as pd
import openpyxl
import numpy as np
import os

In [57]:
# ==============================================================================
# 1. SETUP INICIAL E CARREGAMENTO
#    (Melhor prática: definir constantes e carregar o arquivo de forma segura)
# ==============================================================================

# Definição do caminho do arquivo em uma variável para fácil manutenção
FILE_PATH = 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21.xlsx'
OUTPUT_DIR = 'Dados Gerados'

# Verifica se o arquivo existe antes de tentar carregá-lo
if not os.path.exists(FILE_PATH):
    print(f"ERRO: O arquivo '{FILE_PATH}' não foi encontrado.")
    # Encerra o script ou lança uma exceção se o arquivo principal não existir
    exit()

# Cria a pasta de saída se ela não existir
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Arquivo '{FILE_PATH}' encontrado. Iniciando processamento...")

Arquivo 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21.xlsx' encontrado. Iniciando processamento...


In [58]:
# Listar todas as planilhas disponíveis no arquivo
caminho_arquivo = "Planilha Monitoramento_PPI - Carregamento - BI 2025 - 7.xlsx"
xls = pd.ExcelFile(caminho_arquivo)
sheet_names = xls.sheet_names
sheet_names

['INFORMAÇÕES',
 'PER',
 'PLAN_EXEC (ATÉ 2024)',
 'INEXECUTADO ATÉ 2024 Pendências',
 'A EXECUTAR',
 'META(2025)',
 'META(2026)',
 'META(2027)',
 'Todas Tabelas']

In [59]:
file_path = 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21 - informacao.xlsx'

<h1><b><i>INFORMAÇÕES (1)</b></i></h1>

In [60]:
# %% =====================================================================
# LEITURA INTELIGENTE DA PLANILHA "INFORMAÇÕES (1)" – VERSÃO HIERÁRQUICA
# ========================================================================

import pandas as pd
import numpy as np
from collections import defaultdict

def expandir_mescladas_para_cabecalho(df_cab):
    """Expande horizontalmente (ffill) para simular células mescladas."""
    return df_cab.ffill(axis=1)

def montar_cabecalho_hierarquico(df_cab):
    """
    Recebe um DataFrame com N linhas de cabeçalho (já com ffill)
    e retorna uma lista de nomes de colunas concatenando os níveis.
    """
    lixo = {"#REF!", "TOTAL", "SOMA", 0, "0", None, ""}
    df_cab = df_cab.replace(list(lixo), pd.NA)
    
    colunas = []
    for col in range(df_cab.shape[1]):
        partes = []
        for row in range(df_cab.shape[0]):
            valor = df_cab.iat[row, col]
            if pd.isna(valor):
                continue
            valor = str(valor).strip()
            if not valor or valor.startswith(("=", "#")):
                continue
            if valor.upper() in ("TOTAL", "SOMA"):
                continue
            try:
                float(valor.replace(",", "."))
                continue
            except:
                pass
            partes.append(valor)
        colunas.append(" - ".join(partes) if partes else "")
    
    # Remove duplicatas adicionando sufixo __n
    contador = defaultdict(int)
    novas = []
    for nome in colunas:
        contador[nome] += 1
        if contador[nome] == 1:
            novas.append(nome)
        else:
            novas.append(f"{nome}__{contador[nome]}")
    return novas

# ========================================================================
# 1. DETECTAR A ÚLTIMA COLUNA COM DADOS NO CABEÇALHO
# ========================================================================

file_path = 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21 - informacao.xlsx'
sheet_name = 'INFORMAÇÕES (1)'

# Lê apenas as 4 primeiras linhas para identificar a última coluna não vazia
# (o cabeçalho de INFORMAÇÕES tem umas 3-4 linhas úteis)
cabecalho_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None, nrows=4)

# Encontra o índice da última coluna que contém algum valor não nulo (em qualquer linha)
ultima_coluna_idx = cabecalho_raw.columns[cabecalho_raw.notna().any(axis=0)].max() + 1
if pd.isna(ultima_coluna_idx):
    ultima_coluna_idx = 0

num_colunas = int(ultima_coluna_idx)
print(f"🔍 Última coluna com dados: {num_colunas}")

# ========================================================================
# 2. LER CABEÇALHO E DADOS COM O MESMO NÚMERO DE COLUNAS
# ========================================================================

cabecalho_raw = pd.read_excel(
    file_path, sheet_name=sheet_name, header=None, nrows=4, usecols=range(num_colunas)
)

# Expande horizontalmente (simula mesclas)
cabecalho_expandido = expandir_mescladas_para_cabecalho(cabecalho_raw)

# Gera os nomes das colunas
novos_nomes = montar_cabecalho_hierarquico(cabecalho_expandido)

# Lê os dados pulando as 4 primeiras linhas
df_informacoes = pd.read_excel(
    file_path, sheet_name=sheet_name, header=None, skiprows=4, usecols=range(num_colunas)
)

# Atribui os nomes gerados
df_informacoes.columns = novos_nomes

# Remove linhas totalmente vazias
df_informacoes = df_informacoes.dropna(how='all')

print(f"✅ Planilha INFORMAÇÕES carregada com {df_informacoes.shape[0]} linhas e {df_informacoes.shape[1]} colunas.")

# ========================================================================
# 3. IDENTIFICAR COLUNAS DE IDENTIFICAÇÃO E PREENCHER PARA BAIXO
# ========================================================================

KEYWORDS_ID = [
    "SETOR", "UF", "ESTADO/LOTE", "BR", "EMPREENDIMENTO", "PROPONENTE",
    "EXECUTOR", "ESTRUTURADOR", "ANO LEILÃO", "DATA DE INÍCIO",
    "ANO DA CONCESSÃO", "PRAZO", "CAPEX", "OPEX", "INVESTIMENTO TOTAL",
    "TRECHO", "KM INICIAL", "KM FINAL", "EXTENSÃO", "SITUAÇÃO"
]

id_cols = []
for col in df_informacoes.columns:
    col_upper = col.upper()
    if any(kw in col_upper for kw in KEYWORDS_ID):
        id_cols.append(col)

print("Colunas identificadas como ID:")
for c in id_cols:
    print(f"  - {c}")

# Aplica ffill apenas nessas colunas
if id_cols:
    df_informacoes[id_cols] = df_informacoes[id_cols].ffill()
    print("✅ Forward fill aplicado nas colunas de identificação.")

# ========================================================================
# 4. REMOVER LINHAS COM "TOTAL" OU "SOMA" NAS COLUNAS DE MÉTRICA
# ========================================================================

metric_cols = [c for c in df_informacoes.columns if c not in id_cols]

if metric_cols:
    mask_total = df_informacoes[metric_cols].astype(str).apply(
        lambda row: row.str.contains('TOTAL|SOMA', case=False, na=False).any(), axis=1
    )
    df_informacoes = df_informacoes[~mask_total].reset_index(drop=True)

print(f"✅ Linhas com TOTAL/SOMA removidas. Restam {df_informacoes.shape[0]} linhas.")

# ========================================================================
# 5. REORGANIZAR COLUNAS E CRIAR ID-ÚNICO
# ========================================================================

# Remover colunas que são totalmente vazias (se houver)
df_informacoes = df_informacoes.dropna(axis=1, how='all')

# Renomear colunas que tenham nomes muito longos ou duplicados (ajuste manual se necessário)
# Por exemplo, se aparecer "KM INICIAL" e "KM FINAL" com sufixos, podemos limpar.
# Aqui fazemos uma limpeza simples: remove sufixos __n se não forem necessários
# (mas mantemos se houver duplicatas reais)
df_informacoes.columns = [col.replace("__", "_") for col in df_informacoes.columns]

# Criar ID-ÚNICO (índice começando em 1)
df_informacoes.index = range(1, len(df_informacoes) + 1)
df_informacoes.index.name = 'ID-ÚNICO'
df_informacoes.reset_index(inplace=True)

# ========================================================================
# 6. (OPCIONAL) SALVAR
# ========================================================================

df_informacoes.to_excel("Dados Gerados/INFORMACOES_Processada.xlsx", index=False)
print("✅ Arquivo salvo.")

🔍 Última coluna com dados: 20
✅ Planilha INFORMAÇÕES carregada com 162 linhas e 20 colunas.
Colunas identificadas como ID:
  - SETOR
  - UF
  - PLANILHA DE MONITORAMENTO - ESTADO/LOTE
  - PLANILHA DE MONITORAMENTO - BR
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - EMPREENDIMENTO
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - PROPONENTE
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - EXECUTOR             (Grupo Controlador)
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - ESTRUTURADOR DO PROJETO
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - ANO LEILÃO
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - DATA DE INÍCIO
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - ANO DA CONCESSÃO
  - INFORMAÇÕES CONTRATUAIS - DADOS CONTRATUAIS - PRAZO (anos)
  - INFORMAÇÕES CONTRATUAIS - DADOS CONTRATUAIS - CAPEX (BI) - CAPEX TOTAL:
  - INFORMAÇÕES CONTRATUAIS - DADOS CONTRATUAIS - OPEX (BI) - OPEX TOTAL:
  - INFORMAÇÕES CONTRATUAIS - DADOS CONTRATUAIS - INVESTIMENTO TOTAL (